In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# BigQuery DataFrames - Productionize Mode (Experimental)

This notebook introduces the experimental **productionize** feature in BigQuery DataFrames (`bigframes`). 

BigQuery DataFrames is great for interactive exploration, data science, and quick prototyping. However, once you want to schedule or productionize a data pipeline, running Python scripts or notebook cells directly can be inefficient or expensive due to serverless execution of local code. 

**Productionize mode** addresses this by letting you define your end-to-end data pipeline using the familiar pandas-like BigQuery DataFrames API, while completely deferring execution. Instead of executing queries immediately, BigQuery DataFrames records the operations, traces data flow dependencies, and compiles the entire pipeline into:
1. **A single, optimized SQL script** that can be executed natively in BigQuery.
2. **A Dataform project**, enabling software engineering best practices (version control, dependencies, testing, environments) for your SQL pipelines.

---

### Key Concepts demonstrated in this notebook:
1. **Deferred Execution & Verification**: Run operations within a block where immediate execution or downloading is blocked.
2. **Query Parameters**: Use `bpd.parameter` to define dynamic runtime parameters that will be compiled into SQL/Dataform variables.
3. **User Defined Functions (UDFs)**: Define custom scalar logic with `@bpd.udf` and have it automatically compiled and packaged.
4. **Target Compilation**: Generate a pure BigQuery script (`.to_sql()`) or export a full Dataform project structure (`.export_dataform()`).


## Setup & Import Libraries

First, let's install and import the necessary libraries. Note that `productionize` is in preview/experimental mode.


In [ ]:
import bigframes.pandas as bpd

# Initialize the BigQuery DataFrames session
# We configure the project explicitly to 'bigframes-dev' for this demo
bpd.options.bigquery.project = "bigframes-dev"
session = bpd.get_global_session()
project_id = session.bqclient.project
print(f"Using Google Cloud Project: {project_id}")

## Defining a Productionized Pipeline

When we enter a `bpd.productionize()` context block, BigQuery DataFrames enters a special recording state:
* Calls to load data or perform transformations are deferred and recorded.
* Operations that require pulling data back to the client environment (like `.to_pandas()`, `.head()`, `.shape`, or `.peek()`) will throw a `ProductionizeBlockedError` because they cannot be statically compiled.
* Writing data with `.to_gbq()` is recorded as a pipeline output target rather than executing immediately.

Let's define a pipeline that:
1. Accepts a parameter `min_body_mass` to dynamically filter the input.
2. Applies a Python User-Defined Function (UDF) to transform the data (packaged into a BigQuery persistent routine).
3. Reads from the public `penguins` dataset.
4. Outputs the final filtered and transformed results to a BigQuery table.


In [ ]:
# Ensure the destination dataset exists before we build the pipeline
DATASET_ID = "bigframes_productionize_demo"
session.bqclient.create_dataset(DATASET_ID, exists_ok=True)

# 1. Start the productionize context block
with bpd.productionize() as pipeline:
    # 2. Define a runtime parameter for filtering
    min_body_mass = bpd.parameter("min_body_mass", dtype=float)

    # 3. Define a custom Python UDF that converts grams to kilograms
    @bpd.udf(input_types=[float], output_type=float, dataset=DATASET_ID, name="g_to_kg")
    def g_to_kg(grams):
        if grams is None:
            return None
        return grams / 1000.0

    # 4. Read from the public penguins table
    df = bpd.read_gbq("bigquery-public-data.ml_datasets.penguins")

    # 5. Apply transformations: filter by parameter and call the UDF
    df_filtered = df[df["body_mass_g"] >= min_body_mass]
    df_transformed = df_filtered.copy()
    df_transformed["body_mass_kg"] = g_to_kg(df_filtered["body_mass_g"])

    # Select columns we want to save
    result = df_transformed[["species", "island", "body_mass_g", "body_mass_kg"]]

    # 6. Record writing the output to our destination table
    result.to_gbq(
        f"{project_id}.{DATASET_ID}.processed_penguins",
        if_exists="replace"
    )


## 1. Compiling to a BigQuery SQL Script

Once recorded, we can compile the entire sequence of operations into a single, standard BigQuery SQL script. 

This script:
1. Declares and creates/replaces the registered UDF.
2. Creates or replaces the output table using the optimized compiled query.
3. Automatically references the dynamic `@min_body_mass` parameter.

Let's print the generated SQL.


In [ ]:
# Retrieve compiled SQL
sql_script = pipeline.to_sql()
print(sql_script)

### Executing the Compiled SQL Script

You can execute the compiled SQL script on BigQuery using the standard Google Cloud BigQuery client library, supplying values for any defined parameters:


In [ ]:
from google.cloud import bigquery

# Run the query script using the session's bigquery client
# We supply a parameter value of 4000.0 grams for min_body_mass
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("min_body_mass", "FLOAT64", 4000.0)
    ]
)

print("Executing pipeline SQL script on BigQuery...")
query_job = session.bqclient.query(sql_script, job_config=job_config)
query_job.result()  # Wait for query execution to complete
print("Execution completed successfully!")

# Verify that the destination table contains the filtered and UDF-transformed results
df_result = bpd.read_gbq(f"{project_id}.{DATASET_ID}.processed_penguins")
# Inspect the schema and some results (now outside of productionize mode, so execution is allowed)
print(f"Number of rows: {len(df_result)}")
df_result.head(10)


## 2. Exporting to a Dataform Project

For larger datasets and production environments, [Dataform](https://cloud.google.com/dataform) is the recommended way to model, manage, and schedule SQL workflows in BigQuery. 

`pipeline.export_dataform("path/to/folder")` writes a Dataform package containing:
* A `definitions/` directory containing `.sqlx` files.
* Dependencies between tables and UDF functions are automatically traced and written using Dataform `dependencies: [...]` config blocks.
* Query parameters are converted into Dataform project variables (`${dataform.projectConfig.vars.param}`).
* References to intermediate or source tables are rewritten to Dataform `${ref("table_name")}` calls to allow dynamic multi-environment compilation (e.g. staging vs. production).

Let's export the project to a local folder and inspect the files.


In [ ]:
import os
import tempfile

# Create a directory to export the Dataform project
export_dir = os.path.join(tempfile.gettempdir(), "dataform_pipeline")
os.makedirs(export_dir, exist_ok=True)

# Export
print(f"Exporting pipeline to: {export_dir}")
pipeline.export_dataform(export_dir)

# List the files in the directory
print("\nGenerated files:")
for root, _, files in os.walk(export_dir):
    for file in files:
        rel_path = os.path.join(os.path.relpath(root, export_dir), file)
        print(f"- {rel_path}")


### Inspecting the Generated Dataform Files

Let's read and display the content of the generated UDF and table files to see how the parameters, UDFs, and dependencies are translated to Dataform syntax:


In [ ]:
udf_file = os.path.join(export_dir, "definitions", "g_to_kg.sqlx")
if os.path.exists(udf_file):
    print("=== definitions/g_to_kg.sqlx ===")
    with open(udf_file, "r") as f:
        print(f.read())

table_file = os.path.join(export_dir, "definitions", "processed_penguins.sqlx")
if os.path.exists(table_file):
    print("\n=== definitions/processed_penguins.sqlx ===")
    with open(table_file, "r") as f:
        print(f.read())


### Deploying to Google Cloud Dataform

You can upload the exported pipeline files directly to a Google Cloud Dataform workspace programmatically using the `google-cloud-dataform` library.

The following code demonstrates how to:
1. Initialize a `DataformClient`.
2. Locate (or create) a Dataform workspace in your project.
3. Write/update files from your local exported directory to the remote Dataform workspace.
4. Trigger compilation of the workspace code.

*Note: Before running the cell below, ensure you have the `google-cloud-dataform` library installed (`pip install google-cloud-dataform`) and that your active Google Cloud credentials have the necessary permissions (e.g. `roles/dataform.admin` or `roles/dataform.editor`).*


In [ ]:
# Configure your Dataform repository and workspace details:
# Region must match the location of your Dataform repository (e.g., 'us-central1' or 'us')
REGION_ID = "us-central1"
REPOSITORY_ID = "my-dataform-repo"
WORKSPACE_ID = "bigframes-pipeline-workspace"

# Construct the repository and workspace paths
repo_name = f"projects/{project_id}/locations/{REGION_ID}/repositories/{REPOSITORY_ID}"
workspace_name = f"{repo_name}/workspaces/{WORKSPACE_ID}"

# We comment out the active API calls below so they don't block notebook execution
# if a Dataform repository hasn't been set up yet. Simply uncomment to deploy!

# from google.cloud import dataform_v1beta1
# import os

# try:
#     dataform_client = dataform_v1beta1.DataformClient()

#     # Get or create the workspace
#     try:
#         dataform_client.get_workspace(name=workspace_name)
#         print(f"Using existing Dataform workspace: {workspace_name}")
#     except Exception:
#         print(f"Creating new Dataform workspace: {WORKSPACE_ID}")
#         dataform_client.create_workspace(
#             request=dataform_v1beta1.CreateWorkspaceRequest(
#                 parent=repo_name,
#                 workspace_id=WORKSPACE_ID,
#             )
#         )

#     # Upload files to the workspace
#     print("Uploading exported pipeline files to workspace...")
#     for root, _, files in os.walk(export_dir):
#         for file in files:
#             local_path = os.path.join(root, file)
#             relative_path = os.path.relpath(local_path, export_dir)

#             with open(local_path, "r", encoding="utf-8") as f:
#                 contents = f.read()

#             # Write/overwrite file in the Dataform workspace
#             dataform_client.write_file(
#                 request=dataform_v1beta1.WriteFileRequest(
#                     workspace=workspace_name,
#                     path=relative_path,
#                     contents=contents.encode("utf-8"),
#                 )
#             )
#     print("Upload completed successfully!")

#     # Compile the workspace
#     print("Compiling workspace...")
#     compilation_result = dataform_client.create_compilation_result(
#         request=dataform_v1beta1.CreateCompilationResultRequest(
#             parent=repo_name,
#             compilation_result=dataform_v1beta1.CompilationResult(
#                 workspace=workspace_name
#             ),
#         )
#     )
#     if compilation_result.compilation_errors:
#         print("Dataform compilation failed with errors:")
#         for error in compilation_result.compilation_errors:
#             print(f"- {error.path}: {error.message}")
#     else:
#         print("Dataform compilation succeeded!")

# except Exception as e:
#     print("Deployment failed. Ensure google-cloud-dataform is installed and credentials/repository config is correct.")
#     print(f"Error details: {e}")
